# Step 2 — Sentinel-1 flood mask

Spec: `CLAUDE.md` §6 step 2 (as amended 2026-09-16).

**This measures flood detection, not loss.** There is no field-level loss data: units can be
classified flooded or not, never lost or not. The 1,919-acre county figure is an aggregate from a
voluntary survey with no map.

- Pre window 1 Feb – 8 Mar 2023, post window 11–20 Mar 2023, IW GRD, **VV and VH**, one orbit
  direction, pre paired with post on the **same relative orbit**.
- 3×3 focal mean for speckle. Per pixel, per polarization: flooded if the change is < −3 dB **and**
  the post value is below an absolute threshold. Masks reported VV-only, VH-only and combined.
- Permanent water removed (`JRC/GSW1_4/GlobalSurfaceWater` occurrence > 50).
- **Baseline wetness is checked before differencing**, because the pre window sits in a wet winter
  that included the January flooding.
- Comparison against the Step 1c optical reference in both directions, against the 15 March extent
  and the full union, with the optical 0.1 index-minimum variant carried as a **sensitivity band**.
- **River stage** from the USGS gauges in `data/raw/usgs/`, since the breach was a hydraulic
  failure rather than a rain-on-field event.

**CHECKPOINT:** pre image, post image and flood mask side by side with Pajaro labelled; the human
confirms the breach area shows as flooded and the Salinas Valley does not.

In [ ]:
import ee
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap

import common as C
import optical
from maps import HALO, add_north_arrow, add_scale_bar, ee_tif

ee.Initialize(project=C.EE_PROJECT)

PEAK = pd.Timestamp("2023-03-11")
PRE_WINDOW = ("2023-02-01", "2023-03-09")    # 1 Feb - 8 Mar 2023 inclusive
POST_WINDOW = ("2023-03-11", "2023-03-21")   # 11-20 Mar 2023 inclusive
AOI_LL = (-121.90, 36.60, -121.55, 36.98)
DIFF_DB = -3.0        # spec: change threshold, both polarizations
ABS_VV_DB = -15.0     # spec: absolute threshold, calibrated for VV
ABS_VH_DB = -22.0     # variant: VH sits well below VV over land, see the note in section 3
ACRE_M2 = 4046.8564224

counties_fc = ee.FeatureCollection(C.COUNTIES).filter(ee.Filter.inList("GEOID", C.COUNTY_GEOIDS))
region = counties_fc.geometry()
counties = gpd.GeoDataFrame.from_features(counties_fc.getInfo()["features"], crs="EPSG:4326")
counties = counties[["GEOID", "NAME", "geometry"]].sort_values("GEOID").reset_index(drop=True)
counties_m = counties.to_crs(C.GRID_CRS)
towns = gpd.GeoDataFrame({"name": list(C.TOWNS)}, crs="EPSG:4326",
                         geometry=gpd.points_from_xy(*zip(*C.TOWNS.values()))).to_crs(C.GRID_CRS)

fields = ee.FeatureCollection(C.UNITS_ASSET)
units_fc = fields.filter(ee.Filter.eq("is_unit", 1))
units_tbl = pd.read_csv(C.DERIVED / "01_units_dwr.csv")
N_UNITS = int(units_tbl.is_unit.sum())
perm_water = ee.Image(C.GSW).select("occurrence").unmask(0).gt(C.PERMANENT_WATER)


def stamp(window, extra=""):
    print(f"n units = {N_UNITS} | date window: {window} | datasets: {C.S1}, {C.GSW}, "
          f"{C.UNITS_ASSET}{extra}")


def acres(mask, geom=None, scale=20):
    v = (ee.Image.pixelArea().updateMask(mask.selfMask())
         .reduceRegion(ee.Reducer.sum(), geom or region, scale=scale, maxPixels=1e10, tileScale=8)
         .get("area").getInfo())
    return (v or 0) / ACRE_M2


print(f"n units = {N_UNITS} | pre {PRE_WINDOW[0]} to 2023-03-08 | post {POST_WINDOW[0]} to 2023-03-20")

## 1. Sentinel-1 acquisitions and the choice of relative orbit

In [ ]:
s1 = (ee.ImageCollection(C.S1).filterBounds(region)
      .filter(ee.Filter.eq("instrumentMode", "IW"))
      .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
      .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH")))


def inventory(col, label):
    rows = col.map(lambda im: ee.Feature(None, {
        "date": im.date().format("YYYY-MM-dd"),
        "pass": im.get("orbitProperties_pass"),
        "rel_orbit": im.get("relativeOrbitNumber_start"),
    })).getInfo()["features"]
    df = pd.DataFrame([r["properties"] for r in rows])
    df["window"] = label
    return df.drop_duplicates()


pre_inv = inventory(s1.filterDate(*PRE_WINDOW), "pre")
post_inv = inventory(s1.filterDate(*POST_WINDOW), "post")
inv = pd.concat([pre_inv, post_inv], ignore_index=True)
inv["rel_orbit"] = inv.rel_orbit.astype(int)
print("Sentinel-1 IW scenes with VV and VH over the study area:")
print(inv.groupby(["window", "pass", "rel_orbit"]).date.agg(["count", "min", "max"]).to_string())

post_by_track = post_inv.assign(rel_orbit=post_inv.rel_orbit.astype(int)).groupby(
    ["pass", "rel_orbit"]).date.min().reset_index()
pre_counts = pre_inv.assign(rel_orbit=pre_inv.rel_orbit.astype(int)).groupby(
    ["pass", "rel_orbit"]).date.nunique().rename("pre_scenes").reset_index()
cand = post_by_track.merge(pre_counts, on=["pass", "rel_orbit"], how="left").fillna({"pre_scenes": 0})
cand["days_after_peak"] = (pd.to_datetime(cand.date) - PEAK).dt.days
cand = cand[cand.pre_scenes >= 2].sort_values(["days_after_peak", "pass"])
print("\nCandidate tracks (>= 2 pre scenes), by how soon after the peak they were acquired:")
print(cand.to_string(index=False))

TRACK = cand.iloc[0]
PASS, REL_ORBIT, POST_DATE = TRACK["pass"], int(TRACK.rel_orbit), TRACK.date
print(f"\nChosen: {PASS} relative orbit {REL_ORBIT}, post acquisition {POST_DATE} "
      f"({int(TRACK.days_after_peak)} day(s) after the peak), {int(TRACK.pre_scenes)} pre scenes.")
stamp(f"pre {PRE_WINDOW[0]} to 2023-03-08; post {POST_WINDOW[0]} to 2023-03-20")

## 2. Baseline wetness, before any differencing

The pre window sits in a wet winter that already included the January flooding. Water-like
backscatter here means the change detection starts from an already-wet surface and will understate
the March event.

In [ ]:
track = s1.filter(ee.Filter.eq("orbitProperties_pass", PASS)).filter(
    ee.Filter.eq("relativeOrbitNumber_start", REL_ORBIT))
kernel = ee.Kernel.square(radius=1, units="pixels")
pre_img = track.filterDate(*PRE_WINDOW).select(["VV", "VH"]).median().focal_mean(kernel=kernel)
post_img = (track.filterDate(POST_DATE, ee.Date(POST_DATE).advance(1, "day"))
            .select(["VV", "VH"]).mosaic().focal_mean(kernel=kernel))

base_vv = pre_img.select("VV").lt(ABS_VV_DB).And(perm_water.Not())
base_vh = pre_img.select("VH").lt(ABS_VH_DB).And(perm_water.Not())
study_acres = acres(ee.Image(1).clip(region))
rows = [{"baseline water-like": f"VV < {ABS_VV_DB:g} dB", "acres": acres(base_vv),
         "% of study area": 100 * acres(base_vv) / study_acres},
        {"baseline water-like": f"VH < {ABS_VH_DB:g} dB", "acres": acres(base_vh),
         "% of study area": 100 * acres(base_vh) / study_acres}]
print(pd.DataFrame(rows).round(2).to_string(index=False))

base_frac = (base_vv.rename("base_vv").addBands(base_vh.rename("base_vh"))
             .reduceRegions(units_fc, ee.Reducer.mean(), scale=10, tileScale=8))
bf = pd.DataFrame([f["properties"] for f in base_frac.select(
    ["unit_id", "base_vv", "base_vh"], retainGeometry=False).getInfo()["features"]])
print(f"\nUnits with water-like baseline backscatter over more than 10% of the field: "
      f"VV {int((bf.base_vv > 0.1).sum())} of {len(bf)}, VH {int((bf.base_vh > 0.1).sum())} of {len(bf)}")
print(f"Median baseline water-like fraction per unit: VV {bf.base_vv.median():.3f}, VH {bf.base_vh.median():.3f}")
bf.to_csv(C.DERIVED / "02_baseline_wetness_units.csv", index=False)
print("Wrote 02_baseline_wetness_units.csv")
stamp(f"pre {PRE_WINDOW[0]} to 2023-03-08")

## 3. Flood masks: VV, VH and combined

The spec's absolute threshold of −15 dB is calibrated for VV. Over land VH typically sits several dB
lower, so −15 dB is not restrictive for VH and a VH-only mask at that threshold is close to a
change-only mask. Both are reported: VH at the spec's −15 dB and VH at −22 dB, which is the
conventional open-water level for this band.

In [ ]:
d_vv = post_img.select("VV").subtract(pre_img.select("VV"))
d_vh = post_img.select("VH").subtract(pre_img.select("VH"))
flood_vv = d_vv.lt(DIFF_DB).And(post_img.select("VV").lt(ABS_VV_DB))
flood_vh_spec = d_vh.lt(DIFF_DB).And(post_img.select("VH").lt(ABS_VV_DB))
flood_vh = d_vh.lt(DIFF_DB).And(post_img.select("VH").lt(ABS_VH_DB))
MASKS = {
    "VV only (spec)": flood_vv,
    "VH only (spec thresholds)": flood_vh_spec,
    "VH only (VH -22 dB)": flood_vh,
    "VV and VH (both)": flood_vv.And(flood_vh),
    "VV or VH (either)": flood_vv.Or(flood_vh),
}
MASKS = {k: v.And(perm_water.Not()).rename("flood") for k, v in MASKS.items()}

mask_area = pd.DataFrame([{"mask": k, "acres": acres(v)} for k, v in MASKS.items()])
print(mask_area.round(0).to_string(index=False))
print(f"\nPost acquisition {POST_DATE}, {PASS} relative orbit {REL_ORBIT}; change < {DIFF_DB:g} dB; "
      f"permanent water removed.")
stamp(f"pre {PRE_WINDOW[0]} to 2023-03-08; post {POST_DATE}")

## 4. Per-unit flooded fraction

In [ ]:
def per_unit(mask, name, fc):
    out = []
    ids = fc.aggregate_array("unit_id").getInfo()
    for i in range(0, len(ids), 400):
        chunk = fc.filter(ee.Filter.inList("unit_id", ids[i:i + 400]))
        red = mask.reduceRegions(chunk, ee.Reducer.mean(), scale=10, tileScale=8)
        out += [f["properties"] for f in red.select(["unit_id", "flood"], retainGeometry=False)
                .getInfo()["features"]]
    return pd.DataFrame(out).rename(columns={"flood": name})


flood_units = units_tbl[units_tbl.is_unit.astype(bool)][["unit_id", "COUNTY", "ACRES", "sequence"]].copy()
flood_units["unit_id"] = flood_units.unit_id.astype(str)
for name, key in [("s1_vv", "VV only (spec)"), ("s1_vh", "VH only (VH -22 dB)"),
                  ("s1_both", "VV and VH (both)")]:
    part = per_unit(MASKS[key], name, units_fc)
    part["unit_id"] = part.unit_id.astype(str)
    flood_units = flood_units.merge(part, on="unit_id", how="left")

for col in ["s1_vv", "s1_vh", "s1_both"]:
    flood_units[col] = flood_units[col].fillna(0)
    flood_units[col + "_acres"] = flood_units[col] * flood_units.ACRES

summary = pd.DataFrame([{
    "mask": col,
    "units with fraction > 0.3": int((flood_units[col] > 0.3).sum()),
    "acres in those units": flood_units.loc[flood_units[col] > 0.3, "ACRES"].sum(),
    "area-weighted flooded acres": flood_units[col + "_acres"].sum(),
} for col in ["s1_vv", "s1_vh", "s1_both"]])
print(summary.round(0).to_string(index=False))
print(f"\nCounty aggregate for comparison only: 1,919 acres of strawberries reported destroyed or "
      f"unable to be planted (Monterey County Agricultural Commissioner, 2023-05-12; "
      f"docs/ground_truth_aggregate.md). That is a loss figure from a voluntary survey with no map; "
      f"this table is flood detection over {N_UNITS} units totalling {flood_units.ACRES.sum():,.0f} acres.")
flood_units.to_csv(C.DERIVED / "02_units_s1_flood.csv", index=False)
print("Wrote 02_units_s1_flood.csv")
stamp(f"post {POST_DATE}", extra=", docs/ground_truth_aggregate.md")

## 5. Agreement with the Step 1c optical reference

Both directions, against the 15 March extent and the full union, at the default optical index
minimum 0.0 with the 0.1 variant as a sensitivity band (shown in brackets).

In [ ]:
pre_idx = optical.baseline(region)
opt = {
    "15 March extent": (optical.water_on(region, optical.DATES[0], pre_idx, 0.0),
                        optical.water_on(region, optical.DATES[0], pre_idx, 0.1)),
    "union (15+20+25 March)": (optical.union_water(region, pre=pre_idx, idx_min=0.0),
                               optical.union_water(region, pre=pre_idx, idx_min=0.1)),
}
assert abs(acres(opt["union (15+20+25 March)"][0]) - 6189) < 60, "optical union differs from Step 1c"

rows = []
for s1_name in ["VV only (spec)", "VH only (VH -22 dB)", "VV and VH (both)"]:
    s1m = MASKS[s1_name]
    s1_ac = acres(s1m)
    for opt_name, (o0, o1) in opt.items():
        band = []
        for o in (o0, o1):
            o_ac, both_ac = acres(o), acres(s1m.And(o))
            band.append((100 * both_ac / s1_ac if s1_ac else np.nan,
                         100 * both_ac / o_ac if o_ac else np.nan, o_ac))
        rows.append({
            "S1 mask": s1_name, "optical reference": opt_name,
            "S1 acres": s1_ac, "optical acres": f"{band[0][2]:,.0f} [{band[1][2]:,.0f}]",
            "% of S1 inside optical": f"{band[0][0]:.1f} [{band[1][0]:.1f}]",
            "% of optical inside S1": f"{band[0][1]:.1f} [{band[1][1]:.1f}]",
        })
agree = pd.DataFrame(rows)
print(agree.to_string(index=False, formatters={"S1 acres": "{:,.0f}".format}))
print("\nBrackets are the sensitivity band: the optical reference recomputed with index minima at 0.1.")
print("Neither mask is truth. The optical reference under-counts (no observation 11-14 March); the "
      "radar mask is a threshold rule on one acquisition.")
agree.to_csv(C.DERIVED / "02_s1_optical_agreement.csv", index=False)
print("Wrote 02_s1_optical_agreement.csv")
stamp(f"S1 post {POST_DATE}; optical 15/20/25 March", extra=f", {C.S2}")

## 6. River stage and discharge (USGS gauges)

In [ ]:
dv = pd.read_csv(C.RAW / "usgs" / "usgs_dv_2023-01-01_2023-03-31.csv")
ev = dv[(dv.date >= "2023-03-09") & (dv.date <= "2023-03-20")]
jan = dv[(dv.date >= "2023-01-04") & (dv.date <= "2023-01-16")]
rows = []
for site, g in ev.groupby("site_no"):
    j = jan[jan.site_no == site]
    i = g.value.idxmax()
    rows.append({"site": site, "name": g.site_name.iloc[0], "parameter": g.parameter.iloc[0],
                 "March peak": g.loc[i, "value"], "on": g.loc[i, "date"],
                 "January peak (context)": j.value.max() if len(j) else np.nan})
print(pd.DataFrame(rows).to_string(index=False))

iv_path = C.RAW / "usgs" / "usgs_iv_2023-03-08_2023-03-22.csv"
if iv_path.exists():
    iv = pd.read_csv(iv_path)
    peaks = iv.loc[iv.groupby(["site_no", "parameter"]).value.idxmax()]
    print("\nInstantaneous peaks, 8-22 March 2023:")
    print(peaks[["site_no", "site_name", "parameter", "value", "timestamp"]].to_string(index=False))
else:
    print("\nNo instantaneous-value file; daily values only.")
print("\nThe Pajaro breach was a hydraulic failure: water reached these fields because a levee gave "
      "way upstream, not because a given amount of rain fell on the field. Rainfall totals come in "
      "Step 4 and are compared with these peaks there.")
print(f"n units = {N_UNITS} | date window: 2023-03-09 to 2023-03-20 (January 4-16 as context) | "
      f"datasets: USGS daily and instantaneous values, data/raw/usgs/")

## 7. Figure 2 — pre, post, flood mask

In [ ]:
def db_vis(img, band):
    return img.select(band).visualize(min=-25, max=0, palette=["000000", "ffffff"])


pre_arr, ext = ee_tif(db_vis(pre_img, "VV"), C.DERIVED / "02_pre_vv.tif", AOI_LL, 20)
post_arr, _ = ee_tif(db_vis(post_img, "VV"), C.DERIVED / f"02_post_vv_{POST_DATE}.tif", AOI_LL, 20)
mask_arr, _ = ee_tif(MASKS["VV only (spec)"].unmask(0).toByte(),
                     C.DERIVED / f"02_flood_vv_{POST_DATE}.tif", AOI_LL, 20)
opt_arr, _ = ee_tif(opt["union (15+20+25 March)"][0].unmask(0).toByte(),
                    C.DERIVED / "02_optical_union_20m.tif", AOI_LL, 20)

fig, axes = plt.subplots(1, 3, figsize=(16, 8.2))
for ax, arr, title in [
        (axes[0], pre_arr, f"(a) pre: VV median, {PRE_WINDOW[0]} to 2023-03-08"),
        (axes[1], post_arr, f"(b) post: VV, {POST_DATE}"),
        (axes[2], post_arr, f"(c) flood mask (red) and optical reference (cyan)")]:
    ax.imshow(arr, extent=ext, origin="upper", cmap="gray" if arr.ndim == 2 else None)
    counties_m.boundary.plot(ax=ax, color="yellow", lw=0.7)
    for _, t in towns.iterrows():
        ax.plot(t.geometry.x, t.geometry.y, marker="o", ms=5, color="cyan", mec="black")
        ax.annotate(t["name"], (t.geometry.x, t.geometry.y), xytext=(6, 6), textcoords="offset points",
                    color="cyan", fontsize=9, fontweight="bold", path_effects=HALO)
    ax.set_xlim(ext[0], ext[1])
    ax.set_ylim(ext[2], ext[3])
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title, fontsize=10)
    add_scale_bar(ax, 5)
    add_north_arrow(ax)

axes[2].imshow(np.ma.masked_equal(opt_arr, 0), extent=ext, origin="upper",
               cmap=ListedColormap(["cyan"]), alpha=0.55, interpolation="nearest")
axes[2].imshow(np.ma.masked_equal(mask_arr, 0), extent=ext, origin="upper",
               cmap=ListedColormap(["red"]), interpolation="nearest")
axes[2].plot([], [], "s", color="red", label=f"Sentinel-1 flood, VV ({mask_area.loc[mask_area['mask'] == 'VV only (spec)', 'acres'].iloc[0]:,.0f} ac)")
axes[2].plot([], [], "s", color="cyan", label="optical reference union")
axes[2].legend(loc="lower left", fontsize=8, framealpha=0.95)

fig.suptitle(f"Figure 2. Sentinel-1 flood mask, {PASS.lower()} relative orbit {REL_ORBIT}, "
             f"post {POST_DATE}", fontsize=13, fontweight="bold")
fig.text(0.01, 0.005,
         f"Sources: {C.S1} IW GRD, VV; pre = median {PRE_WINDOW[0]} to 2023-03-08, post = {POST_DATE}, "
         f"same relative orbit; 3x3 focal mean; flooded if change < {DIFF_DB:g} dB and post < {ABS_VV_DB:g} dB; "
         f"permanent water {C.GSW} occurrence > {C.PERMANENT_WATER} removed.\n"
         f"Optical reference: Step 1c union of 15, 20 and 25 March ({C.S2}). Flood detection, not loss. "
         f"EPSG:3310, displayed at 20 m.",
         fontsize=8, va="bottom")
fig.tight_layout(rect=(0, 0.05, 1, 0.95))
FIG = C.FIGURES / "fig2_s1_flood_mask.png"
fig.savefig(FIG, dpi=200)
plt.show()
print(f"Saved {FIG.name}")
stamp(f"pre {PRE_WINDOW[0]} to 2023-03-08; post {POST_DATE}", extra=f", {C.S2}")